Step 0 - Explanation

In [ ]:
"""
Preprocessing Notebook: FASTA -> TXT -> Masked TXT
---------------------------------------------------
Reads Salmonella_1.fna to Salmonella_6.fna from the current directory.

Step 1 - FASTA to TXT:
- Salmonella_1 to Salmonella_5 --> combined into train.txt
- Salmonella_6                  --> test.txt

Step 2 - Masking (5-mer rotation):
- train.txt --> train_m.txt
- test.txt  --> test_m.txt

Masking Strategy:
- Slides a 5-mer window across the sequence
- Masks 1 position per window in a rotating pattern (pos 1,2,3,4,5,1,2,...)
- Masked positions are replaced with 'M'
- Original files are kept as targets for training
"""

'\nPreprocessing Script: FASTA -> TXT\n-----------------------------------\nReads Salmonella_1.fna to Salmonella_6.fna from a folder.\n- Salmonella_1 to Salmonella_5 --> combined into train.txt\n- Salmonella_6                  --> test.txt\n \nUsage:\n    python preprocess_fasta.py --input_dir /path/to/fasta/folder --output_dir /path/to/output\n'

Step 1 - Libraries

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np

Step 2 - Main code

In [2]:
def parse_fasta(filepath: str) -> str:
    sequence_parts = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                continue
            sequence_parts.append(line.upper())
    return "".join(sequence_parts)

def main():
    class args:
        input_dir  = "."
        output_dir = "."

    os.makedirs(args.output_dir, exist_ok=True)

    train_files = [f"Salmonella_{i}.fna" for i in range(1, 6)]
    test_file   = "Salmonella_6.fna"

    train_output = os.path.join(args.output_dir, "train.txt")
    total_train_bases = 0

    with open(train_output, "w") as out_f:
        for fname in train_files:
            fpath = os.path.join(args.input_dir, fname)
            if not os.path.exists(fpath):
                raise FileNotFoundError(f"Expected file not found: {fpath}")
            seq = parse_fasta(fpath)
            out_f.write(seq + "\n")
            total_train_bases += len(seq)
            print(f"  [train] {fname}: {len(seq):,} bases")

    print(f"\n  Training file written  --> {train_output}")
    print(f"  Total training bases   : {total_train_bases:,}\n")

    test_output = os.path.join(args.output_dir, "test.txt")
    fpath = os.path.join(args.input_dir, test_file)
    if not os.path.exists(fpath):
        raise FileNotFoundError(f"Expected file not found: {fpath}")
    seq = parse_fasta(fpath)

    with open(test_output, "w") as out_f:
        out_f.write(seq + "\n")

    print(f"  [test]  {test_file}: {len(seq):,} bases")
    print(f"  Test file written      --> {test_output}")
    print(f"\nDone! train.txt and test.txt ready — proceeding to masking.")

main()

  [train] Salmonella_1.fna: 5,203,495 bases
  [train] Salmonella_2.fna: 4,952,765 bases
  [train] Salmonella_3.fna: 4,638,880 bases
  [train] Salmonella_4.fna: 4,951,383 bases
  [train] Salmonella_5.fna: 4,831,756 bases

  Training file written  --> .\train.txt
  Total training bases   : 24,578,279

  [test]  Salmonella_6.fna: 4,857,466 bases
  Test file written      --> .\test.txt

Done! train.txt and test.txt ready — proceeding to masking.


In [3]:
def mask_sequence(seq: str) -> str:
    """Apply 5-mer rotation masking to a sequence.
    Cycles through positions 1,2,3,4,5,1,2,3,4,5...
    and masks that position within each 5-mer window."""
    seq = list(seq)
    rotation = [0, 1, 2, 3, 4]  # positions to mask, 0-indexed
    
    num_windows = len(seq) // 5
    for i in range(num_windows):
        mask_pos = rotation[i % 5]
        seq[i * 5 + mask_pos] = 'M'
    
    return "".join(seq)


def mask_file(input_path: str, output_path: str):
    """Read a txt file, mask each genome line, write to output."""
    with open(input_path, "r") as in_f, open(output_path, "w") as out_f:
        for line in in_f:
            seq = line.strip()
            masked = mask_sequence(seq)
            out_f.write(masked + "\n")
            total = len(seq)
            masked_count = masked.count('M')
            print(f"  Bases: {total:,} | Masked: {masked_count:,} ({masked_count/total*100:.1f}%)")


mask_file("train.txt", "train_m.txt")
print("  train_m.txt written\n")

mask_file("test.txt", "test_m.txt")
print("  test_m.txt written\n")

print("Done! All 4 files ready: train.txt, train_m.txt, test.txt, test_m.txt")

  Bases: 5,203,495 | Masked: 1,040,699 (20.0%)
  Bases: 4,952,765 | Masked: 990,553 (20.0%)
  Bases: 4,638,880 | Masked: 927,776 (20.0%)
  Bases: 4,951,383 | Masked: 990,276 (20.0%)
  Bases: 4,831,756 | Masked: 966,351 (20.0%)
  train_m.txt written

  Bases: 4,857,466 | Masked: 971,493 (20.0%)
  test_m.txt written

Done! All 4 files ready: train.txt, train_m.txt, test.txt, test_m.txt
